In [ ]:
from pathlib import Path

WORK_ROOT = Path("/mnt/data/agentic_repo_runs")

RUN_LABEL = "repo_runtime_v26"

REPO_SOURCE = ""

REPO_REF = ""

TASK_PROMPT = ""

ISSUE_CONTEXT = ""

LLM_BACKEND = "llama_cpp"

LLAMA_MODEL_PATH = ""

LLAMA_CONTEXT_LENGTH = 8192

LLAMA_GPU_LAYERS = -1

LLAMA_TEMPERATURE = 0.2

OPENAI_COMPAT_BASE_URL = ""

OPENAI_COMPAT_MODEL = ""

OPENAI_COMPAT_API_KEY = ""

MAX_ATTEMPTS = 3

CANDIDATES_PER_ATTEMPT = 4

CIRCUIT_CANDIDATES_PER_ATTEMPT = 4

TOP_REPO_CHUNKS = 20

MAX_FILE_CHARS = 12000

ENABLE_GIT_JOURNAL = True

AUTO_COMMIT_CANDIDATES = False

ENABLE_QNODE_SHAPE_ORACLE = True

ENABLE_SPECTRAL_GENOME = True

ENABLE_COLOR_HARMONICS = True

ENABLE_BENCHMARK_LAB = True

ENABLE_MENTOR_PARLIAMENT = True

ENABLE_CIRCUIT_WEATHER_MAP = True

ENABLE_PATCH_BIOMES = True

ENABLE_SHAPE_SENTINEL = True

ENABLE_REPO_TEACHING_LAYER = True

ENABLE_PALETTE_DRIFT_LEDGER = True

ENABLE_RETRIEVAL_CONSTELLATION = True

ENABLE_REPO_MEMORY_BRAID = True

GENOME_SELECTION_TOP_K = 3

GENOME_MUTATION_RATE = 0.12

HARMONIC_DRIFT_TOLERANCE = 0.35

TEST_COMMAND_HINT = ""

BENCHMARK_COMMAND_HINT = ""

ALLOW_DELETE_OPS = False


In [ ]:
from __future__ import annotations

import ast

import copy

import dataclasses

import difflib

import hashlib

import io

import json

import math

import os

import re

import shutil

import subprocess

import sys

import textwrap

import time

import zipfile

from collections import Counter, defaultdict

from dataclasses import dataclass, field

from datetime import datetime, timezone

from pathlib import Path

from typing import Any, Dict, Iterable, List, Optional, Tuple

try:

    import requests

except Exception:

    requests = None

def now_ts() -> str:

    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def ensure_dir(path: Path) -> Path:

    path.mkdir(parents=True, exist_ok=True)

    return path

def safe_slug(value: str, default: str = "item") -> str:

    cleaned = re.sub(r"[^a-zA-Z0-9._-]+", "-", (value or "").strip()).strip("-._")

    return cleaned[:80] or default

def read_text(path: Path, default: str = "") -> str:

    try:

        return path.read_text(encoding="utf-8")

    except Exception:

        return default

def write_text(path: Path, text: str) -> Path:

    ensure_dir(path.parent)

    path.write_text(text, encoding="utf-8")

    return path

def write_json(path: Path, data: Any) -> Path:

    ensure_dir(path.parent)

    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")

    return path

def sha256_text(text: str) -> str:

    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def sha256_file(path: Path) -> str:

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(lambda: f.read(1024 * 1024), b""):

            h.update(chunk)

    return h.hexdigest()

def pretty_json(data: Any) -> str:

    return json.dumps(data, indent=2, ensure_ascii=False)

def run_cmd(cmd: str, cwd: Optional[Path] = None, timeout: int = 600) -> Dict[str, Any]:

    started = time.time()

    proc = subprocess.run(

        cmd,

        cwd=str(cwd) if cwd else None,

        shell=True,

        text=True,

        capture_output=True,

        timeout=timeout,

    )

    return {

        "cmd": cmd,

        "cwd": str(cwd) if cwd else None,

        "returncode": proc.returncode,

        "stdout": proc.stdout[-40000:],

        "stderr": proc.stderr[-40000:],

        "elapsed_sec": round(time.time() - started, 3),

    }

def copytree_durable(src: Path, dst: Path, ignore_git: bool = False) -> None:

    if dst.exists():

        shutil.rmtree(dst)

    ignore = shutil.ignore_patterns(".git") if ignore_git else None

    shutil.copytree(src, dst, ignore=ignore)

def zip_dir(src_dir: Path, zip_path: Path) -> Path:

    ensure_dir(zip_path.parent)

    if zip_path.exists():

        zip_path.unlink()

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:

        for path in sorted(src_dir.rglob("*")):

            if path.is_file():

                zf.write(path, arcname=str(path.relative_to(src_dir)))

    return zip_path

def is_text_like(path: Path) -> bool:

    text_exts = {

        ".py", ".md", ".txt", ".json", ".yaml", ".yml", ".toml", ".ini", ".cfg",

        ".js", ".ts", ".tsx", ".jsx", ".css", ".scss", ".html", ".xml", ".sh",

        ".bat", ".ps1", ".java", ".kt", ".go", ".rs", ".c", ".cpp", ".h", ".hpp",

        ".sql", ".r", ".ipynb"

    }

    return path.suffix.lower() in text_exts or path.name in {

        "Dockerfile", "Makefile", "requirements.txt", "pyproject.toml", "package.json"

    }

def git_available() -> bool:

    try:

        subprocess.run(["git", "--version"], capture_output=True, check=False)

        return True

    except Exception:

        return False

def normalize_ws(text: str) -> str:

    return re.sub(r"\s+", " ", text or "").strip()

def clipped(text: str, n: int = 8000) -> str:

    text = text or ""

    return text if len(text) <= n else text[:n] + "\n...[clipped]..."

def relative_paths(root: Path) -> List[str]:

    out: List[str] = []

    for p in sorted(root.rglob("*")):

        if p.is_file():

            out.append(str(p.relative_to(root)))

    return out

def repo_size_summary(root: Path) -> Dict[str, Any]:

    total_files = 0

    total_bytes = 0

    ext_counter: Counter[str] = Counter()

    for p in root.rglob("*"):

        if p.is_file():

            total_files += 1

            try:

                total_bytes += p.stat().st_size

            except Exception:

                pass

            ext_counter[p.suffix.lower() or "<no_ext>"] += 1

    return {

        "total_files": total_files,

        "total_bytes": total_bytes,

        "top_extensions": ext_counter.most_common(20),

    }

def infer_python_imports(text: str) -> List[str]:

    found: List[str] = []

    try:

        tree = ast.parse(text)

    except Exception:

        return found

    for node in ast.walk(tree):

        if isinstance(node, ast.Import):

            for alias in node.names:

                found.append(alias.name)

        elif isinstance(node, ast.ImportFrom):

            if node.module:

                found.append(node.module)

    return found

def find_keyword_windows(text: str, keywords: List[str], radius: int = 20) -> List[str]:

    lines = text.splitlines()

    hits: List[Tuple[int, int]] = []

    lower = [k.lower() for k in keywords if k.strip()]

    if not lower:

        return ["\n".join(lines[: min(len(lines), 120)])]

    for i, line in enumerate(lines):

        low = line.lower()

        if any(k in low for k in lower):

            start = max(0, i - radius)

            end = min(len(lines), i + radius)

            hits.append((start, end))

    if not hits:

        return ["\n".join(lines[: min(len(lines), 120)])]

    merged: List[Tuple[int, int]] = []

    for start, end in sorted(hits):

        if not merged or start > merged[-1][1] + 5:

            merged.append([start, end])

        else:

            merged[-1][1] = max(merged[-1][1], end)

    return ["\n".join(lines[s:e]) for s, e in merged[:4]]

@dataclass

class RuntimeState:

    root: Path

    artifacts_dir: Path = field(init=False)

    logs_dir: Path = field(init=False)

    repo_origin_dir: Path = field(init=False)

    repo_work_dir: Path = field(init=False)

    candidates_dir: Path = field(init=False)

    best_repo_dir: Path = field(init=False)

    memory_dir: Path = field(init=False)

    artifacts: List[str] = field(default_factory=list)

    events: List[Dict[str, Any]] = field(default_factory=list)

    def __post_init__(self) -> None:

        self.artifacts_dir = ensure_dir(self.root / "artifacts")

        self.logs_dir = ensure_dir(self.root / "logs")

        self.repo_origin_dir = self.root / "repo_origin"

        self.repo_work_dir = self.root / "repo_work"

        self.candidates_dir = ensure_dir(self.root / "candidate_runs")

        self.best_repo_dir = self.root / "best_repo"

        self.memory_dir = ensure_dir(self.root / "memory")

    def emit_json(self, relative: str, data: Any) -> Path:

        path = write_json(self.artifacts_dir / relative, data)

        self.artifacts.append(str(path.relative_to(self.root)))

        return path

    def emit_text(self, relative: str, text: str) -> Path:

        path = write_text(self.artifacts_dir / relative, text)

        self.artifacts.append(str(path.relative_to(self.root)))

        return path

    def log_event(self, kind: str, **payload: Any) -> None:

        event = {"ts": now_ts(), "kind": kind, **payload}

        self.events.append(event)

        write_json(self.logs_dir / "events.json", self.events)

    def snapshot_repo(self, src: Path, name: str, ignore_git: bool = False) -> Path:

        dst = self.candidates_dir / safe_slug(name)

        copytree_durable(src, dst, ignore_git=ignore_git)

        return dst

def assert_repo_source_configured() -> None:

    if not (REPO_SOURCE or "").strip():

        raise ValueError("REPO_SOURCE is empty. Set a git URL or local repo path in the top cell.")

def materialize_repo(state: RuntimeState) -> Path:

    assert_repo_source_configured()

    src = (REPO_SOURCE or "").strip()

    dest = state.repo_origin_dir

    if dest.exists():

        shutil.rmtree(dest)

    if re.match(r"^(https?://|git@|ssh://)", src):

        if not git_available():

            raise RuntimeError("git is required to clone a remote repository.")

        run = run_cmd(f'git clone "{src}" "{dest}"', timeout=1800)

        state.emit_json("repo_clone_result.json", run)

        if run["returncode"] != 0:

            raise RuntimeError(f"git clone failed:\n{run['stderr']}")

    else:

        src_path = Path(src).expanduser().resolve()

        if not src_path.exists():

            raise FileNotFoundError(f"Local repo path not found: {src_path}")

        copytree_durable(src_path, dest, ignore_git=False)

        state.emit_json("repo_copy_result.json", {"source": str(src_path), "dest": str(dest)})

    if (REPO_REF or "").strip() and git_available() and (dest / ".git").exists():

        checkout = run_cmd(f'git checkout "{REPO_REF.strip()}"', cwd=dest, timeout=600)

        state.emit_json("repo_checkout_result.json", checkout)

        if checkout["returncode"] != 0:

            raise RuntimeError(f"git checkout failed:\n{checkout['stderr']}")

    copytree_durable(dest, state.repo_work_dir, ignore_git=False)

    return state.repo_work_dir

def maybe_create_branch(repo_dir: Path, branch_name: str) -> Optional[Dict[str, Any]]:

    if not ENABLE_GIT_JOURNAL or not git_available() or not (repo_dir / ".git").exists():

        return None

    cmd = f'git checkout -B "{branch_name}"'

    return run_cmd(cmd, cwd=repo_dir, timeout=300)

def maybe_commit(repo_dir: Path, message: str) -> Optional[Dict[str, Any]]:

    if not AUTO_COMMIT_CANDIDATES or not git_available() or not (repo_dir / ".git").exists():

        return None

    run_cmd("git add -A", cwd=repo_dir, timeout=300)

    return run_cmd(f'git commit -m "{message.replace(chr(34), chr(39))}"', cwd=repo_dir, timeout=300)

def repo_file_records(repo_dir: Path, max_file_chars: int = 12000) -> List[Dict[str, Any]]:

    records: List[Dict[str, Any]] = []

    for p in sorted(repo_dir.rglob("*")):

        if not p.is_file():

            continue

        rel = str(p.relative_to(repo_dir))

        if any(part.startswith(".git") for part in p.parts):

            continue

        try:

            size = p.stat().st_size

        except Exception:

            size = 0

        rec: Dict[str, Any] = {"path": rel, "size": size, "suffix": p.suffix.lower()}

        if is_text_like(p):

            txt = read_text(p)

            rec["preview"] = clipped(txt, max_file_chars)

            rec["sha256"] = sha256_text(txt)

            if p.suffix.lower() == ".py":

                rec["imports"] = infer_python_imports(txt)[:120]

            low = txt.lower()

            rec["mentions_pennylane"] = ("pennylane" in low) or ("qml." in low)

            rec["mentions_color"] = any(k in low for k in ["color", "colour", "chroma", "palette", "mix"])

        records.append(rec)

    return records

def infer_repo_profile(repo_dir: Path, records: List[Dict[str, Any]]) -> Dict[str, Any]:

    ext_counter = Counter(r["suffix"] for r in records)

    imports_counter = Counter()

    pennylane_files: List[str] = []

    color_files: List[str] = []

    tests: List[str] = []

    configs: Dict[str, bool] = {

        "pyproject.toml": (repo_dir / "pyproject.toml").exists(),

        "requirements.txt": (repo_dir / "requirements.txt").exists(),

        "package.json": (repo_dir / "package.json").exists(),

        "Makefile": (repo_dir / "Makefile").exists(),

        "tox.ini": (repo_dir / "tox.ini").exists(),

        "pytest.ini": (repo_dir / "pytest.ini").exists(),

    }

    for r in records:

        for imp in r.get("imports", []):

            imports_counter[imp.split(".")[0]] += 1

        if r.get("mentions_pennylane"):

            pennylane_files.append(r["path"])

        if r.get("mentions_color"):

            color_files.append(r["path"])

        if any(tok in r["path"].lower() for tok in ["test", "tests/"]):

            tests.append(r["path"])

    if configs["package.json"]:

        pkg_raw = read_text(repo_dir / "package.json")

        try:

            pkg = json.loads(pkg_raw)

            deps = list((pkg.get("dependencies") or {}).keys()) + list((pkg.get("devDependencies") or {}).keys())

        except Exception:

            deps = []

    else:

        deps = []

    readme = ""

    for name in ["README.md", "readme.md", "README.txt"]:

        p = repo_dir / name

        if p.exists():

            readme = clipped(read_text(p), 12000)

            break

    return {

        "repo_size": repo_size_summary(repo_dir),

        "top_extensions": ext_counter.most_common(20),

        "top_import_roots": imports_counter.most_common(40),

        "pennylane_files": pennylane_files[:80],

        "color_files": color_files[:80],

        "test_files": tests[:120],

        "config_files": configs,

        "package_json_dependencies": deps[:120],

        "readme_excerpt": readme,

        "likely_python_repo": ext_counter[".py"] > 0,

        "likely_js_repo": ext_counter[".js"] + ext_counter[".ts"] + ext_counter[".tsx"] > 0,

    }

def build_repo_index(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:

    index: List[Dict[str, Any]] = []

    for r in records:

        preview = r.get("preview", "")

        keywords = set(re.findall(r"[A-Za-z_][A-Za-z0-9_]{2,}", preview[:5000]))

        score_bias = 0

        if r.get("mentions_pennylane"):

            score_bias += 3

        if r.get("mentions_color"):

            score_bias += 2

        if "test" in r["path"].lower():

            score_bias += 2

        index.append({

            "path": r["path"],

            "suffix": r.get("suffix"),

            "size": r.get("size"),

            "mentions_pennylane": r.get("mentions_pennylane", False),

            "mentions_color": r.get("mentions_color", False),

            "keywords": sorted(list(keywords))[:120],

            "score_bias": score_bias,

        })

    return index

def retrieve_focus_chunks(repo_dir: Path, records: List[Dict[str, Any]], task_text: str, top_k: int = 18) -> List[Dict[str, Any]]:

    tokens = [t.lower() for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]{2,}", task_text or "")]

    scored: List[Tuple[float, Dict[str, Any]]] = []

    for r in records:

        preview = r.get("preview")

        if not preview:

            continue

        score = 0.0

        low_path = r["path"].lower()

        low_prev = preview.lower()

        for tok in tokens:

            if tok in low_path:

                score += 5.0

            score += low_prev.count(tok) * 0.5

        if r.get("mentions_pennylane") and any(k in tokens for k in ["quantum", "pennylane", "qnode", "circuit", "qml"]):

            score += 6.0

        if r.get("mentions_color") and any(k in tokens for k in ["color", "colour", "palette", "chroma", "mix"]):

            score += 4.0

        if "test" in low_path:

            score += 1.0

        if score > 0:

            scored.append((score, r))

    scored.sort(key=lambda x: (-x[0], x[1]["path"]))

    chosen = [rec for _, rec in scored[:top_k]]

    if not chosen:

        chosen = [r for r in records if r.get("preview")][: min(top_k, len(records))]

    chunks: List[Dict[str, Any]] = []

    for r in chosen:

        windows = find_keyword_windows(r.get("preview", ""), tokens, radius=18)

        for idx, window in enumerate(windows[:2]):

            chunks.append({

                "path": r["path"],

                "chunk_id": f'{r["path"]}::chunk{idx+1}',

                "content": clipped(window, 5000),

                "mentions_pennylane": r.get("mentions_pennylane", False),

                "mentions_color": r.get("mentions_color", False),

            })

    return chunks[: top_k * 2]

def repo_diff_summary(before_dir: Path, after_dir: Path) -> Dict[str, Any]:

    before_paths = set(relative_paths(before_dir))

    after_paths = set(relative_paths(after_dir))

    added = sorted(after_paths - before_paths)

    removed = sorted(before_paths - after_paths)

    maybe_changed = sorted(before_paths & after_paths)

    changed: List[str] = []

    patches: Dict[str, str] = {}

    for rel in maybe_changed:

        a = before_dir / rel

        b = after_dir / rel

        if a.is_file() and b.is_file() and is_text_like(a) and is_text_like(b):

            ta = read_text(a)

            tb = read_text(b)

            if ta != tb:

                changed.append(rel)

                diff = "\n".join(

                    difflib.unified_diff(

                        ta.splitlines(),

                        tb.splitlines(),

                        fromfile=f"a/{rel}",

                        tofile=f"b/{rel}",

                        lineterm="",

                    )

                )

                patches[rel] = clipped(diff, 30000)

        else:

            try:

                if sha256_file(a) != sha256_file(b):

                    changed.append(rel)

            except Exception:

                changed.append(rel)

    return {

        "added": added,

        "removed": removed,

        "changed": changed,

        "patches": patches,

        "changed_count": len(changed),

        "added_count": len(added),

        "removed_count": len(removed),

    }

def safe_join_repo(repo_dir: Path, rel_path: str) -> Path:

    candidate = (repo_dir / rel_path).resolve()

    repo_root = repo_dir.resolve()

    if not str(candidate).startswith(str(repo_root)):

        raise ValueError(f"Unsafe repo-relative path: {rel_path}")

    return candidate

def apply_patch_plan(repo_dir: Path, plan: Dict[str, Any]) -> Dict[str, Any]:

    ops = plan.get("file_operations") or []

    applied: List[Dict[str, Any]] = []

    errors: List[str] = []

    for i, op in enumerate(ops, start=1):

        kind = (op.get("op") or "").strip()

        rel_path = (op.get("path") or "").strip()

        try:

            target = safe_join_repo(repo_dir, rel_path) if rel_path else repo_dir

            if kind == "mkdir":

                ensure_dir(target)

            elif kind == "write_file":

                content = op.get("content", "")

                write_text(target, content)

            elif kind == "append_file":

                ensure_dir(target.parent)

                with target.open("a", encoding="utf-8") as f:

                    f.write(op.get("content", ""))

            elif kind == "ensure_contains":

                content = read_text(target, "")

                needle = op.get("needle", "")

                insertion = op.get("content", needle)

                if needle not in content:

                    if content and not content.endswith("\n"):

                        content += "\n"

                    content += insertion

                    write_text(target, content)

            elif kind == "replace_text":

                content = read_text(target, "")

                old = op.get("old", "")

                new = op.get("new", "")

                if old not in content:

                    raise ValueError(f"replace_text target not found in {rel_path}")

                write_text(target, content.replace(old, new, op.get("count", -1)))

            elif kind == "insert_after":

                content = read_text(target, "")

                anchor = op.get("anchor", "")

                if anchor not in content:

                    raise ValueError(f"insert_after anchor not found in {rel_path}")

                new_block = anchor + op.get("content", "")

                write_text(target, content.replace(anchor, new_block, 1))

            elif kind == "delete_path":

                if not ALLOW_DELETE_OPS:

                    raise ValueError("delete_path blocked because ALLOW_DELETE_OPS=False")

                if target.is_dir():

                    shutil.rmtree(target)

                elif target.exists():

                    target.unlink()

            else:

                raise ValueError(f"Unsupported patch op: {kind}")

            applied.append({"index": i, "op": kind, "path": rel_path})

        except Exception as e:

            errors.append(f"op#{i} {kind} {rel_path}: {e}")

    return {"applied": applied, "errors": errors, "applied_count": len(applied), "error_count": len(errors)}


In [ ]:
from __future__ import annotations

CHROMATIC_BASES: Dict[str, Dict[str, float]] = {

    "planner":            {"red": 0.18, "green": 0.34, "blue": 0.48, "gold": 0.22, "violet": 0.26},

    "implementer":        {"red": 0.31, "green": 0.46, "blue": 0.28, "gold": 0.18, "violet": 0.19},

    "judge":              {"red": 0.24, "green": 0.25, "blue": 0.36, "gold": 0.38, "violet": 0.21},

    "quantum_architect":  {"red": 0.12, "green": 0.29, "blue": 0.57, "gold": 0.30, "violet": 0.42},

    "chroma_mentor":      {"red": 0.19, "green": 0.39, "blue": 0.31, "gold": 0.41, "violet": 0.34},

    "anomaly_witness":    {"red": 0.41, "green": 0.21, "blue": 0.22, "gold": 0.17, "violet": 0.11},

}

CHROMATIC_ACTIONS = {

    "red": "bold refactor, structural change, aggressive cleanup",

    "green": "stability, compatibility, test-centric repair",

    "blue": "architecture, abstraction, interface cleanup",

    "gold": "observability, docs, mentor alignment, human readability",

    "violet": "quantum nuance, unusual edge-case handling, advanced capability",

}

@dataclass

class ChromaticEpisode:

    attempt_index: int

    candidate_index: int

    role: str

    palette: Dict[str, float]

    rationale: str

    mentor_notes: List[str] = field(default_factory=list)

    anomaly_pressure: float = 0.0

    quantum_bias: float = 0.0

    def to_dict(self) -> Dict[str, Any]:

        return dataclasses.asdict(self)

def normalize_palette(p: Dict[str, float]) -> Dict[str, float]:

    total = sum(max(v, 0.0) for v in p.values()) or 1.0

    return {k: round(max(v, 0.0) / total, 6) for k, v in p.items()}

def blend_palettes(*palettes: Dict[str, float], weights: Optional[List[float]] = None) -> Dict[str, float]:

    if not palettes:

        return normalize_palette(copy.deepcopy(CHROMATIC_BASES["planner"]))

    if weights is None:

        weights = [1.0] * len(palettes)

    acc: Dict[str, float] = defaultdict(float)

    for palette, w in zip(palettes, weights):

        for k, v in palette.items():

            acc[k] += v * w

    return normalize_palette(dict(acc))

def palette_distance(a: Dict[str, float], b: Dict[str, float]) -> float:

    keys = sorted(set(a) | set(b))

    return round(sum(abs(a.get(k, 0.0) - b.get(k, 0.0)) for k in keys), 6)

def dominant_channels(palette: Dict[str, float], top_n: int = 3) -> List[str]:

    return [k for k, _ in sorted(palette.items(), key=lambda kv: (-kv[1], kv[0]))[:top_n]]

def palette_story(palette: Dict[str, float]) -> str:

    chans = dominant_channels(palette, 3)

    parts = [f"{c}: {CHROMATIC_ACTIONS.get(c, c)}" for c in chans]

    return "; ".join(parts)

def candidate_palette(attempt_index: int, candidate_index: int, quantum_boost: float = 0.0) -> Dict[str, float]:

    base = blend_palettes(

        CHROMATIC_BASES["planner"],

        CHROMATIC_BASES["implementer"],

        CHROMATIC_BASES["judge"],

        CHROMATIC_BASES["quantum_architect"],

        weights=[1.0, 1.15 + candidate_index * 0.05, 0.9, 0.8 + quantum_boost],

    )

    drift = {

        "red": 0.01 * attempt_index,

        "green": 0.025 * candidate_index,

        "blue": 0.03 * quantum_boost,

        "gold": 0.015 * (attempt_index + candidate_index),

        "violet": 0.04 * quantum_boost + 0.01 * attempt_index,

    }

    return normalize_palette({k: base.get(k, 0.0) + drift.get(k, 0.0) for k in base})

def mentor_palette_adjustment(notes: List[str], has_quantum: bool, has_color: bool) -> Dict[str, float]:

    base = copy.deepcopy(CHROMATIC_BASES["chroma_mentor"])

    text = " ".join(notes).lower()

    if has_quantum or "quantum" in text or "qml" in text:

        base["violet"] += 0.16

        base["blue"] += 0.06

    if has_color or "palette" in text or "chroma" in text or "mix" in text:

        base["gold"] += 0.08

        base["green"] += 0.04

    if "risk" in text or "regression" in text or "fragile" in text:

        base["green"] += 0.12

        base["red"] -= 0.03

    if "ambitious" in text or "bold" in text:

        base["red"] += 0.08

    return normalize_palette(base)

def build_chromatic_episode(

    attempt_index: int,

    candidate_index: int,

    mentor_notes: List[str],

    has_quantum: bool,

    has_color: bool,

) -> ChromaticEpisode:

    base = candidate_palette(attempt_index, candidate_index, quantum_boost=1.0 if has_quantum else 0.0)

    mentor = mentor_palette_adjustment(mentor_notes, has_quantum, has_color)

    final_palette = blend_palettes(base, mentor, weights=[1.0, 0.55])

    anomaly_pressure = round(0.08 * attempt_index + 0.04 * candidate_index, 4)

    quantum_bias = round(final_palette.get("violet", 0.0) + final_palette.get("blue", 0.0), 4)

    rationale = palette_story(final_palette)

    return ChromaticEpisode(

        attempt_index=attempt_index,

        candidate_index=candidate_index,

        role="arena_candidate",

        palette=final_palette,

        rationale=rationale,

        mentor_notes=mentor_notes,

        anomaly_pressure=anomaly_pressure,

        quantum_bias=quantum_bias,

    )

def chromatic_memory_capsules(profile: Dict[str, Any], charter: Dict[str, Any]) -> List[Dict[str, Any]]:

    capsules: List[Dict[str, Any]] = []

    if profile.get("pennylane_files"):

        capsules.append({

            "capsule_id": "capsule-quantum-shape",

            "theme": "Protect measurement shapes, qnode signatures, and differentiability paths.",

            "trigger": "When a patch touches qnodes, templates, wires, devices, or gradient settings.",

        })

    if profile.get("color_files"):

        capsules.append({

            "capsule_id": "capsule-chroma-consistency",

            "theme": "Keep color/palette mixing semantics stable and avoid hidden drift in channel balance.",

            "trigger": "When a patch touches palette logic, color transforms, or chromatic mentors.",

        })

    capsules.append({

        "capsule_id": "capsule-election-rationale",

        "theme": "Preserve why a candidate won; keep judge and validation reasoning durable.",

        "trigger": "On every arena election.",

    })

    if (charter.get("task_title") or "").lower().find("test") >= 0:

        capsules.append({

            "capsule_id": "capsule-validation-tightening",

            "theme": "Increase test coverage and preserve direct observable acceptance signals.",

            "trigger": "When candidate validation is close but not decisive.",

        })

    return capsules


In [ ]:
from __future__ import annotations

class BaseLLM:

    backend_name: str = "base"

    def generate(self, system: str, user: str, temperature: float = 0.2, max_tokens: int = 2500) -> str:

        raise NotImplementedError

    def generate_json(self, system: str, user: str, temperature: float = 0.2, max_tokens: int = 2500) -> Dict[str, Any]:

        raw = self.generate(system, user, temperature=temperature, max_tokens=max_tokens)

        parsed = parse_json_response(raw)

        if not isinstance(parsed, dict):

            raise ValueError(f"LLM returned non-object JSON:\n{raw[:4000]}")

        return parsed

def parse_json_response(text: str) -> Any:

    text = (text or "").strip()

    if text.startswith("```"):

        text = re.sub(r"^```(?:json)?\s*", "", text)

        text = re.sub(r"\s*```$", "", text)

    try:

        return json.loads(text)

    except Exception:

        pass

    match = re.search(r"(\{.*\}|\[.*\])", text, flags=re.DOTALL)

    if match:

        return json.loads(match.group(1))

    raise ValueError(f"Could not parse JSON from LLM output:\n{text[:4000]}")

class LlamaCppBackend(BaseLLM):

    backend_name = "llama_cpp"

    def __init__(self, model_path: str, context_length: int = 8192, gpu_layers: int = -1) -> None:

        if not model_path:

            raise ValueError("LLAMA_MODEL_PATH must be set for llama_cpp backend.")

        try:

            from llama_cpp import Llama

        except Exception as e:

            raise RuntimeError("llama_cpp backend selected but llama-cpp-python is not available.") from e

        self._llm = Llama(

            model_path=model_path,

            n_ctx=context_length,

            n_gpu_layers=gpu_layers,

            verbose=False,

        )

    def generate(self, system: str, user: str, temperature: float = 0.2, max_tokens: int = 2500) -> str:

        prompt = (

            "<|system|>\n" + system.strip() + "\n"

            "<|user|>\n" + user.strip() + "\n"

            "<|assistant|>\n"

        )

        out = self._llm(

            prompt,

            max_tokens=max_tokens,

            temperature=temperature,

            stop=["<|user|>", "<|system|>"],

        )

        return out["choices"][0]["text"].strip()

class OpenAICompatibleBackend(BaseLLM):

    backend_name = "openai_compatible"

    def __init__(self, base_url: str, api_key: str, model: str) -> None:

        if requests is None:

            raise RuntimeError("requests is required for the openai_compatible backend.")

        if not base_url or not api_key or not model:

            raise ValueError("OPENAI_COMPAT_BASE_URL, OPENAI_COMPAT_API_KEY, and OPENAI_COMPAT_MODEL are required.")

        self.base_url = base_url.rstrip("/")

        self.api_key = api_key

        self.model = model

    def generate(self, system: str, user: str, temperature: float = 0.2, max_tokens: int = 2500) -> str:

        url = f"{self.base_url}/chat/completions"

        payload = {

            "model": self.model,

            "temperature": temperature,

            "max_tokens": max_tokens,

            "messages": [

                {"role": "system", "content": system},

                {"role": "user", "content": user},

            ],

        }

        headers = {"Authorization": f"Bearer {self.api_key}", "Content-Type": "application/json"}

        res = requests.post(url, headers=headers, json=payload, timeout=300)

        if res.status_code >= 300:

            raise RuntimeError(f"OpenAI-compatible request failed [{res.status_code}]: {res.text[:2000]}")

        data = res.json()

        return data["choices"][0]["message"]["content"].strip()

def build_llm() -> BaseLLM:

    backend = (LLM_BACKEND or "").strip().lower()

    if backend == "llama_cpp":

        return LlamaCppBackend(

            model_path=LLAMA_MODEL_PATH,

            context_length=LLAMA_CONTEXT_LENGTH,

            gpu_layers=LLAMA_GPU_LAYERS,

        )

    if backend == "openai_compatible":

        return OpenAICompatibleBackend(

            base_url=OPENAI_COMPAT_BASE_URL,

            api_key=OPENAI_COMPAT_API_KEY,

            model=OPENAI_COMPAT_MODEL,

        )

    raise ValueError("LLM_BACKEND must be 'llama_cpp' or 'openai_compatible'.")


In [ ]:
from __future__ import annotations

REPO_BRIEF_SYSTEM = """
You are a principal software architect.
Return strict JSON with keys:
- repo_summary: string
- likely_stack: array[string]
- key_subsystems: array[string]
- risky_areas: array[string]
- likely_entrypoints: array[string]
- likely_test_paths: array[string]
- pennylane_assessment: string
- chromatic_assessment: string
- recommended_focus_files: array[string]
Do not include markdown.
"""

TASK_CHARTER_SYSTEM = """
You are a repo task strategist.
Return strict JSON with keys:
- task_title: string
- objective: string
- acceptance_criteria: array[string]
- target_files: array[string]
- constraints: array[string]
- why_now: string
- risk_notes: array[string]
When TASK_PROMPT is blank, invent a repo-grounded task that is concrete, valuable, and testable.
Do not include markdown.
"""

EXECUTION_GRAPH_SYSTEM = """
You are an execution graph planner for a repo-editing agent.
Return strict JSON with keys:
- graph_summary: string
- nodes: array[object] where each object has id, kind, goal, depends_on
- edges: array[object] where each object has from, to, reason
- victory_signals: array[string]
- failure_modes: array[string]
Keep the plan compact and implementable.
Do not include markdown.
"""

VALIDATION_PLAN_SYSTEM = """
You are a validation engineer.
Return strict JSON with keys:
- commands: array[string]
- static_checks: array[string]
- benchmark_commands: array[string]
- observable_signals: array[string]
Use the repo profile and task charter. Prefer commands that are likely to exist.
Do not include markdown.
"""

COLOR_MENTOR_SYSTEM = """
You are the chromatic mentor quorum for an agentic patch arena.
Return strict JSON with keys:
- mentor_notes: array[string]
- palette_goals: array[string]
- drift_warnings: array[string]
- anomaly_watch: array[string]
Balance ambition with regression safety.
Do not include markdown.
"""

CIRCUIT_ARENA_SYSTEM = """
You are a quantum software architect specializing in PennyLane.
Return strict JSON with keys:
- enabled: boolean
- circuit_candidates: array[object] where each object has:
  - name: string
  - motivation: string
  - target_files: array[string]
  - suggested_changes: array[string]
  - validation_ideas: array[string]
  - benchmark_ideas: array[string]
- benchmark_focus: array[string]
- shape_risks: array[string]
Only propose candidates if the repo context suggests PennyLane, qnodes, devices, or quantum workflows.
Do not include markdown.
"""

PATCH_CANDIDATE_SYSTEM = """
You are the implementer in a repo patch arena.
Return strict JSON with keys:
- candidate_name: string
- strategy_summary: string
- file_operations: array[object]
- tests_to_run: array[string]
- expected_observables: array[string]
- risk_mitigation: array[string]
- chromatic_notes: array[string]
- quantum_notes: array[string]
Allowed file operations:
- {"op":"mkdir","path":"dir/subdir"}
- {"op":"write_file","path":"path/to/file","content":"...full file content..."}
- {"op":"append_file","path":"path/to/file","content":"...text..."}
- {"op":"ensure_contains","path":"path/to/file","needle":"exact needle","content":"text to append if missing"}
- {"op":"replace_text","path":"path/to/file","old":"exact old text","new":"replacement text","count":1}
- {"op":"insert_after","path":"path/to/file","anchor":"exact anchor text","content":"text inserted immediately after anchor"}

Rules:
- Prefer surgical edits first.
- If you rewrite a file, include the full new content.
- Touch only files relevant to the task.
- Include tests/docs when useful.
- If code changes are risky, add observability or tests.
Do not include markdown.
"""

JUDGE_SYSTEM = """
You are the final judge for a repo patch arena.
Return strict JSON with keys:
- verdict: string
- score_0_to_10: number
- strengths: array[string]
- weaknesses: array[string]
- acceptance_pass_likelihood: string
- should_continue: boolean
- winner_rationale: string
Score based on repo fit, acceptance criteria coverage, validation results, and regression risk.
Do not include markdown.
"""

PR_SUMMARY_SYSTEM = """
You are a staff engineer writing a concise PR summary.
Return strict JSON with keys:
- title: string
- summary: array[string]
- files_touched: array[string]
- validation: array[string]
- caveats: array[string]
Do not include markdown.
"""

def llm_json_with_artifact(

    state: RuntimeState,

    llm: BaseLLM,

    name: str,

    system: str,

    user: str,

    temperature: float = 0.2,

    max_tokens: int = 2500,

) -> Dict[str, Any]:

    raw = llm.generate(system, user, temperature=temperature, max_tokens=max_tokens)

    state.emit_text(f"raw_llm/{safe_slug(name)}.txt", raw)

    parsed = parse_json_response(raw)

    if not isinstance(parsed, dict):

        raise ValueError(f"Expected JSON object from {name}.")

    state.emit_json(f"structured/{safe_slug(name)}.json", parsed)

    return parsed

def repo_context_text(profile: Dict[str, Any], index: List[Dict[str, Any]], chunks: List[Dict[str, Any]]) -> str:

    return "\n\n".join([

        "REPO PROFILE\n" + pretty_json(profile),

        "REPO INDEX (compact)\n" + pretty_json(index[:80]),

        "FOCUS CHUNKS\n" + pretty_json(chunks[: min(len(chunks), 20)]),

    ])

def select_validation_commands(profile: Dict[str, Any], llm_plan: Dict[str, Any]) -> List[str]:

    commands = []

    if TEST_COMMAND_HINT.strip():

        commands.append(TEST_COMMAND_HINT.strip())

    commands.extend([c for c in (llm_plan.get("commands") or []) if isinstance(c, str) and c.strip()])

    if not commands:

        if profile.get("likely_python_repo"):

            if profile.get("config_files", {}).get("pytest.ini") or profile.get("test_files"):

                commands.append("pytest -q")

            else:

                commands.append("python -m pytest -q")

        if profile.get("likely_js_repo"):

            commands.append("npm test -- --runInBand")

        if not commands:

            commands.append("python -m compileall .")

    deduped: List[str] = []

    seen = set()

    for cmd in commands:

        key = normalize_ws(cmd)

        if key and key not in seen:

            deduped.append(cmd)

            seen.add(key)

    return deduped[:6]

def select_benchmark_commands(profile: Dict[str, Any], llm_plan: Dict[str, Any]) -> List[str]:

    commands = []

    if BENCHMARK_COMMAND_HINT.strip():

        commands.append(BENCHMARK_COMMAND_HINT.strip())

    commands.extend([c for c in (llm_plan.get("benchmark_commands") or []) if isinstance(c, str) and c.strip()])

    if profile.get("pennylane_files") and not commands:

        for path in profile.get("test_files", []):

            low = path.lower()

            if any(tok in low for tok in ["qml", "quantum", "circuit", "qnode", "pennylane"]):

                commands.append(f'pytest "{path}" -q')

                break

    return commands[:4]

def static_file_health(repo_dir: Path) -> Dict[str, Any]:

    broken_py: List[str] = []

    notebook_count = 0

    for p in repo_dir.rglob("*"):

        if p.is_file():

            if p.suffix.lower() == ".py":

                txt = read_text(p)

                try:

                    ast.parse(txt)

                except Exception as e:

                    broken_py.append(f"{p.relative_to(repo_dir)} :: {e}")

            elif p.suffix.lower() == ".ipynb":

                notebook_count += 1

    return {

        "python_parse_errors": broken_py,

        "python_parse_error_count": len(broken_py),

        "notebook_count": notebook_count,

    }

def run_validation_suite(repo_dir: Path, commands: List[str], benchmark_commands: List[str]) -> Dict[str, Any]:

    command_results = [run_cmd(cmd, cwd=repo_dir, timeout=1800) for cmd in commands]

    benchmark_results = [run_cmd(cmd, cwd=repo_dir, timeout=1800) for cmd in benchmark_commands]

    static = static_file_health(repo_dir)

    passed = all(r["returncode"] == 0 for r in command_results) and static["python_parse_error_count"] == 0

    return {

        "commands": command_results,

        "benchmarks": benchmark_results,

        "static": static,

        "passed": passed,

        "passed_benchmarks": all(r["returncode"] == 0 for r in benchmark_results) if benchmark_results else None,

    }

def compact_focus_chunks(chunks: List[Dict[str, Any]], max_chunks: int = 12) -> List[Dict[str, Any]]:

    return chunks[:max_chunks]

def build_patch_prompt(

    repo_brief: Dict[str, Any],

    charter: Dict[str, Any],

    execution_graph: Dict[str, Any],

    validation_plan: Dict[str, Any],

    focus_chunks: List[Dict[str, Any]],

    circuit_pack: Dict[str, Any],

    mentor_pack: Dict[str, Any],

    chroma_episode: ChromaticEpisode,

    prior_attempt_notes: List[str],

) -> str:

    payload = {

        "repo_brief": repo_brief,

        "task_charter": charter,

        "execution_graph": execution_graph,

        "validation_plan": validation_plan,

        "circuit_pack": circuit_pack,

        "mentor_pack": mentor_pack,

        "chromatic_episode": chroma_episode.to_dict(),

        "prior_attempt_notes": prior_attempt_notes[-12:],

        "focus_chunks": compact_focus_chunks(focus_chunks),

    }

    return pretty_json(payload)


In [ ]:
from __future__ import annotations

def summarize_candidate_for_judge(

    patch_plan: Dict[str, Any],

    apply_result: Dict[str, Any],

    diff: Dict[str, Any],

    validation: Dict[str, Any],

    chroma_episode: ChromaticEpisode,

) -> Dict[str, Any]:

    return {

        "candidate_name": patch_plan.get("candidate_name"),

        "strategy_summary": patch_plan.get("strategy_summary"),

        "apply_result": apply_result,

        "diff": {

            "changed_count": diff.get("changed_count"),

            "added_count": diff.get("added_count"),

            "removed_count": diff.get("removed_count"),

            "changed": diff.get("changed", [])[:50],

            "added": diff.get("added", [])[:50],

            "removed": diff.get("removed", [])[:50],

        },

        "validation": validation,

        "chromatic_episode": chroma_episode.to_dict(),

        "risk_mitigation": patch_plan.get("risk_mitigation", []),

        "expected_observables": patch_plan.get("expected_observables", []),

    }

def materialize_pr_summary(state: RuntimeState, llm: BaseLLM, final_record: Dict[str, Any]) -> Dict[str, Any]:

    user = pretty_json(final_record)

    pr = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="pr_summary",

        system=PR_SUMMARY_SYSTEM,

        user=user,

        temperature=0.15,

        max_tokens=1200,

    )

    summary_lines = [f"# {pr.get('title', 'PR Summary')}", ""]

    if pr.get("summary"):

        summary_lines.append("## Summary")

        summary_lines.extend([f"- {x}" for x in pr.get("summary", [])])

        summary_lines.append("")

    if pr.get("files_touched"):

        summary_lines.append("## Files Touched")

        summary_lines.extend([f"- {x}" for x in pr.get("files_touched", [])])

        summary_lines.append("")

    if pr.get("validation"):

        summary_lines.append("## Validation")

        summary_lines.extend([f"- {x}" for x in pr.get("validation", [])])

        summary_lines.append("")

    if pr.get("caveats"):

        summary_lines.append("## Caveats")

        summary_lines.extend([f"- {x}" for x in pr.get("caveats", [])])

        summary_lines.append("")

    state.emit_text("PR_SUMMARY.md", "\n".join(summary_lines).strip() + "\n")

    return pr

def run_agentic_repo_runtime_v24() -> Dict[str, Any]:

    state = RuntimeState(ensure_dir(WORK_ROOT / f"{safe_slug(RUN_LABEL, 'repo_runtime_v24')}_{now_ts()}"))

    state.log_event("runtime_start", version="v24")

    llm = build_llm()

    repo_dir = materialize_repo(state)

    branch_res = maybe_create_branch(repo_dir, f"agentic/{safe_slug(RUN_LABEL, 'runtime-v24')}-{now_ts()}")

    if branch_res:

        state.emit_json("git_branch_result.json", branch_res)

    records = repo_file_records(repo_dir, max_file_chars=MAX_FILE_CHARS)

    profile = infer_repo_profile(repo_dir, records)

    index = build_repo_index(records)

    state.emit_json("repo_manifest.json", records[:8000])

    state.emit_json("repo_profile.json", profile)

    state.emit_json("repo_index.json", index[:8000])

    bootstrap_chunks = retrieve_focus_chunks(repo_dir, records, TASK_PROMPT or ISSUE_CONTEXT or "repo overview quantum color architecture", top_k=TOP_REPO_CHUNKS)

    context_pack = repo_context_text(profile, index, bootstrap_chunks)

    state.emit_text("repo_context_pack.txt", context_pack)

    repo_brief = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="repo_brief",

        system=REPO_BRIEF_SYSTEM,

        user=context_pack,

        temperature=0.15,

        max_tokens=1500,

    )

    charter_prompt = pretty_json({

        "repo_brief": repo_brief,

        "repo_profile": profile,

        "user_task_prompt": TASK_PROMPT,

        "issue_context": ISSUE_CONTEXT,

        "instruction": "If user_task_prompt is blank, invent one valuable task grounded in this repo.",

    })

    task_charter = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="task_charter",

        system=TASK_CHARTER_SYSTEM,

        user=charter_prompt,

        temperature=0.2,

        max_tokens=1800,

    )

    focus_text = " ".join([

        task_charter.get("task_title", ""),

        task_charter.get("objective", ""),

        " ".join(task_charter.get("acceptance_criteria", [])[:20]),

        " ".join(task_charter.get("target_files", [])[:40]),

    ])

    focus_chunks = retrieve_focus_chunks(repo_dir, records, focus_text, top_k=TOP_REPO_CHUNKS)

    state.emit_json("focus_chunks.json", focus_chunks)

    execution_graph = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="execution_graph",

        system=EXECUTION_GRAPH_SYSTEM,

        user=pretty_json({

            "repo_brief": repo_brief,

            "task_charter": task_charter,

            "focus_chunks": compact_focus_chunks(focus_chunks, max_chunks=10),

        }),

        temperature=0.2,

        max_tokens=1800,

    )

    validation_plan = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="validation_plan",

        system=VALIDATION_PLAN_SYSTEM,

        user=pretty_json({

            "repo_profile": profile,

            "task_charter": task_charter,

            "repo_brief": repo_brief,

            "focus_chunks": compact_focus_chunks(focus_chunks, max_chunks=8),

        }),

        temperature=0.1,

        max_tokens=1500,

    )

    mentor_pack = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="chromatic_mentor_bootstrap",

        system=COLOR_MENTOR_SYSTEM,

        user=pretty_json({

            "repo_profile": profile,

            "task_charter": task_charter,

            "repo_brief": repo_brief,

            "execution_graph": execution_graph,

        }),

        temperature=0.25,

        max_tokens=1200,

    )

    circuit_pack = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="circuit_arena_bootstrap",

        system=CIRCUIT_ARENA_SYSTEM,

        user=pretty_json({

            "repo_profile": profile,

            "task_charter": task_charter,

            "focus_chunks": compact_focus_chunks([c for c in focus_chunks if c.get("mentions_pennylane")], max_chunks=10),

        }),

        temperature=0.2,

        max_tokens=1800,

    )

    capsules = chromatic_memory_capsules(profile, task_charter)

    state.emit_json("memory_capsules.json", capsules)

    commands = select_validation_commands(profile, validation_plan)

    benchmark_commands = select_benchmark_commands(profile, validation_plan)

    state.emit_json("resolved_validation_commands.json", {"commands": commands, "benchmark_commands": benchmark_commands})

    base_repo = state.snapshot_repo(repo_dir, "base_repo", ignore_git=False)

    best_record: Optional[Dict[str, Any]] = None

    attempt_notes: List[str] = []

    all_candidates: List[Dict[str, Any]] = []

    for attempt_idx in range(1, MAX_ATTEMPTS + 1):

        mentor_attempt = llm_json_with_artifact(

            state=state,

            llm=llm,

            name=f"chromatic_mentor_attempt_{attempt_idx}",

            system=COLOR_MENTOR_SYSTEM,

            user=pretty_json({

                "attempt_index": attempt_idx,

                "task_charter": task_charter,

                "prior_attempt_notes": attempt_notes[-12:],

                "capsules": capsules,

                "repo_profile": profile,

            }),

            temperature=0.25,

            max_tokens=1200,

        )

        circuit_attempt = llm_json_with_artifact(

            state=state,

            llm=llm,

            name=f"circuit_arena_attempt_{attempt_idx}",

            system=CIRCUIT_ARENA_SYSTEM,

            user=pretty_json({

                "attempt_index": attempt_idx,

                "repo_profile": profile,

                "task_charter": task_charter,

                "prior_attempt_notes": attempt_notes[-12:],

                "focus_chunks": compact_focus_chunks([c for c in focus_chunks if c.get("mentions_pennylane")], max_chunks=10),

            }),

            temperature=0.2,

            max_tokens=1800,

        )

        for candidate_idx in range(1, CANDIDATES_PER_ATTEMPT + 1):

            candidate_repo = state.snapshot_repo(base_repo, f"attempt_{attempt_idx}_candidate_{candidate_idx}", ignore_git=False)

            chroma_episode = build_chromatic_episode(

                attempt_index=attempt_idx,

                candidate_index=candidate_idx,

                mentor_notes=mentor_attempt.get("mentor_notes", []),

                has_quantum=bool(profile.get("pennylane_files")),

                has_color=bool(profile.get("color_files")),

            )

            state.emit_json(

                f"chromatic/attempt_{attempt_idx}_candidate_{candidate_idx}.json",

                chroma_episode.to_dict(),

            )

            patch_plan = llm_json_with_artifact(

                state=state,

                llm=llm,

                name=f"patch_plan_attempt_{attempt_idx}_candidate_{candidate_idx}",

                system=PATCH_CANDIDATE_SYSTEM,

                user=build_patch_prompt(

                    repo_brief=repo_brief,

                    charter=task_charter,

                    execution_graph=execution_graph,

                    validation_plan=validation_plan,

                    focus_chunks=focus_chunks,

                    circuit_pack=circuit_attempt if circuit_attempt.get("enabled") else circuit_pack,

                    mentor_pack=mentor_attempt,

                    chroma_episode=chroma_episode,

                    prior_attempt_notes=attempt_notes,

                ),

                temperature=0.2 + (candidate_idx - 1) * 0.08,

                max_tokens=2800,

            )

            apply_result = apply_patch_plan(candidate_repo, patch_plan)

            state.emit_json(

                f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/apply_result.json",

                apply_result,

            )

            commit_res = maybe_commit(candidate_repo, f"attempt {attempt_idx} candidate {candidate_idx}")

            if commit_res:

                state.emit_json(

                    f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/git_commit_result.json",

                    commit_res,

                )

            diff = repo_diff_summary(base_repo, candidate_repo)

            state.emit_json(

                f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/diff_summary.json",

                {k: v for k, v in diff.items() if k != "patches"},

            )

            state.emit_text(

                f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/patches.diff",

                "\n\n".join([v for _, v in sorted(diff.get("patches", {}).items())])[:200000],

            )

            merged_commands = commands[:]

            for cmd in patch_plan.get("tests_to_run", []):

                if isinstance(cmd, str) and cmd.strip() and normalize_ws(cmd) not in {normalize_ws(x) for x in merged_commands}:

                    merged_commands.append(cmd)

            validation = run_validation_suite(candidate_repo, merged_commands[:6], benchmark_commands)

            state.emit_json(

                f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/validation.json",

                validation,

            )

            judge_input = summarize_candidate_for_judge(

                patch_plan=patch_plan,

                apply_result=apply_result,

                diff=diff,

                validation=validation,

                chroma_episode=chroma_episode,

            )

            judge = llm_json_with_artifact(

                state=state,

                llm=llm,

                name=f"judge_attempt_{attempt_idx}_candidate_{candidate_idx}",

                system=JUDGE_SYSTEM,

                user=pretty_json({

                    "task_charter": task_charter,

                    "acceptance_criteria": task_charter.get("acceptance_criteria", []),

                    "candidate": judge_input,

                }),

                temperature=0.1,

                max_tokens=1200,

            )

            score = float(judge.get("score_0_to_10", 0.0))

            if validation.get("passed"):

                score += 1.25

            if validation.get("passed_benchmarks") is True:

                score += 0.5

            if apply_result.get("error_count", 0) > 0:

                score -= 1.75

            if diff.get("changed_count", 0) + diff.get("added_count", 0) == 0:

                score -= 2.0

            record = {

                "attempt_index": attempt_idx,

                "candidate_index": candidate_idx,

                "candidate_repo": str(candidate_repo),

                "patch_plan": patch_plan,

                "apply_result": apply_result,

                "diff": {k: v for k, v in diff.items() if k != "patches"},

                "validation": validation,

                "judge": judge,

                "chromatic_episode": chroma_episode.to_dict(),

                "composite_score": round(score, 4),

            }

            state.emit_json(

                f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/candidate_record.json",

                record,

            )

            all_candidates.append(record)

        ranked = sorted(all_candidates, key=lambda r: (-r["composite_score"], r["attempt_index"], r["candidate_index"]))

        best_record = ranked[0]

        state.emit_json(f"attempts/attempt_{attempt_idx}/election.json", {

            "attempt_index": attempt_idx,

            "winner_attempt": best_record["attempt_index"],

            "winner_candidate": best_record["candidate_index"],

            "winner_score": best_record["composite_score"],

            "top_5": [

                {

                    "attempt_index": r["attempt_index"],

                    "candidate_index": r["candidate_index"],

                    "composite_score": r["composite_score"],

                    "verdict": r["judge"].get("verdict"),

                }

                for r in ranked[:5]

            ],

        })

        attempt_notes.append(

            f"Attempt {attempt_idx} best candidate was A{best_record['attempt_index']}C{best_record['candidate_index']} "

            f"with score {best_record['composite_score']}. Verdict={best_record['judge'].get('verdict')}."

        )

        attempt_notes.extend(best_record["judge"].get("weaknesses", [])[:4])

        if best_record["validation"].get("passed") and str(best_record["judge"].get("acceptance_pass_likelihood", "")).lower() in {"high", "very high"}:

            break

        base_repo = Path(best_record["candidate_repo"])

    if best_record is None:

        raise RuntimeError("No candidate records were produced.")

    final_repo = Path(best_record["candidate_repo"])

    copytree_durable(final_repo, state.best_repo_dir, ignore_git=False)

    final_zip = zip_dir(state.best_repo_dir, state.root / f"{safe_slug(RUN_LABEL, 'repo_runtime_v24')}_final_repo.zip")

    final_record = {

        "repo_source": REPO_SOURCE,

        "repo_ref": REPO_REF,

        "task_prompt": TASK_PROMPT,

        "issue_context": ISSUE_CONTEXT,

        "repo_profile": profile,

        "repo_brief": repo_brief,

        "task_charter": task_charter,

        "execution_graph": execution_graph,

        "validation_plan": validation_plan,

        "circuit_bootstrap": circuit_pack,

        "chromatic_bootstrap": mentor_pack,

        "best_candidate": best_record,

        "all_candidate_count": len(all_candidates),

        "artifacts_count": len(state.artifacts),

        "final_repo": str(state.best_repo_dir),

        "final_zip": str(final_zip),

    }

    state.emit_json("final_report.json", final_record)

    materialize_pr_summary(state, llm, final_record)

    summary = {

        "run_root": str(state.root),

        "artifacts_dir": str(state.artifacts_dir),

        "candidate_runs_dir": str(state.candidates_dir),

        "best_repo_dir": str(state.best_repo_dir),

        "final_zip": str(final_zip),

        "task_title": task_charter.get("task_title"),

        "best_composite_score": best_record["composite_score"],

        "best_attempt": best_record["attempt_index"],

        "best_candidate": best_record["candidate_index"],

        "validation_passed": best_record["validation"].get("passed"),

        "judge_verdict": best_record["judge"].get("verdict"),

        "judge_acceptance_pass_likelihood": best_record["judge"].get("acceptance_pass_likelihood"),

        "pennylane_files_detected": len(profile.get("pennylane_files", [])),

        "color_files_detected": len(profile.get("color_files", [])),

        "artifact_count": len(state.artifacts),

    }

    state.emit_json("summary.json", summary)

    state.log_event("runtime_complete", **summary)

    return summary


In [ ]:
from __future__ import annotations

MENTOR_PARLIAMENT_SYSTEM = """
You are an expert software-mentorship parliament for an autonomous repo-editing system.
Return strict JSON with keys:
- parliament_summary: string
- patch_budget: object with numeric keys oasis, forge, storm, prism in [0,1]
- member_votes: array[object] where each object has member, stance, favored_biome, cautions, score_bias
- must_have_actions: array[string]
- forbidden_moves: array[string]
- mentor_notes: array[string]
Balance testability, repo fit, explainability, quantum safety, and measured ambition.
Do not include markdown.
"""

REPO_TEACHING_LAYER_SYSTEM = """
You are a staff engineer writing an after-action teaching bundle for a repository change.
Return strict JSON with keys:
- operator_notes: array[string]
- migration_hints: array[string]
- failure_capsules: array[object] where each object has title, lesson, trigger, mitigation
- readme_delta_recommendations: array[string]
- own_this_repo_summary: string
- follow_on_tasks: array[string]
Do not include markdown.
"""

PATCH_BIOMES = ["oasis", "forge", "storm", "prism"]

PATCH_BIOME_GUIDANCE = {

    "oasis": "Make the smallest safe patch that improves acceptance confidence with strong validation leverage.",

    "forge": "Make a balanced functional patch with production-minded code changes and targeted tests.",

    "storm": "Make an ambitious architectural move only when it stays grounded in the retrieved repo context.",

    "prism": "Make a hybrid patch that coordinates quantum nuance, chromatic logic, and developer teachability.",

}

def normalize_patch_budget(budget: Dict[str, Any]) -> Dict[str, float]:

    values = {}

    for biome in PATCH_BIOMES:

        try:

            values[biome] = max(0.0, float((budget or {}).get(biome, 0.0)))

        except Exception:

            values[biome] = 0.0

    total = sum(values.values()) or 1.0

    return {k: round(v / total, 6) for k, v in values.items()}

def assigned_biome(attempt_index: int, candidate_index: int) -> str:

    offset = ((attempt_index - 1) * max(1, CANDIDATES_PER_ATTEMPT) + (candidate_index - 1)) % len(PATCH_BIOMES)

    return PATCH_BIOMES[offset]

def build_mentor_parliament(

    state: RuntimeState,

    llm: BaseLLM,

    repo_profile: Dict[str, Any],

    repo_brief: Dict[str, Any],

    task_charter: Dict[str, Any],

    execution_graph: Dict[str, Any],

    color_harmonics: Dict[str, Any],

    qnode_shape_oracle: Dict[str, Any],

    benchmark_lab: Dict[str, Any],

    prior_attempt_notes: Optional[List[str]] = None,

    name: str = "mentor_parliament",

) -> Dict[str, Any]:

    if not ENABLE_MENTOR_PARLIAMENT:

        return {"enabled": False, "patch_budget": normalize_patch_budget({"oasis": 1, "forge": 1, "storm": 1, "prism": 1}), "member_votes": [], "mentor_notes": []}

    pack = llm_json_with_artifact(

        state=state,

        llm=llm,

        name=name,

        system=MENTOR_PARLIAMENT_SYSTEM,

        user=pretty_json({

            "repo_profile": repo_profile,

            "repo_brief": repo_brief,

            "task_charter": task_charter,

            "execution_graph": execution_graph,

            "color_harmonics": color_harmonics,

            "qnode_shape_oracle": qnode_shape_oracle,

            "benchmark_lab": benchmark_lab,

            "prior_attempt_notes": (prior_attempt_notes or [])[-12:],

        }),

        temperature=0.18,

        max_tokens=1700,

    )

    pack["enabled"] = True

    pack["patch_budget"] = normalize_patch_budget(pack.get("patch_budget", {"oasis": 0.22, "forge": 0.34, "storm": 0.18, "prism": 0.26}))

    pack["favored_biomes"] = [v.get("favored_biome") for v in pack.get("member_votes", []) if isinstance(v, dict) and v.get("favored_biome") in PATCH_BIOMES]

    return pack

def build_circuit_weather_map(repo_dir: Path, profile: Dict[str, Any], records: List[Dict[str, Any]], oracle_report: Dict[str, Any]) -> Dict[str, Any]:

    if not ENABLE_CIRCUIT_WEATHER_MAP:

        return {"enabled": False, "hotspots": [], "files": []}

    qnode_entries = defaultdict(list)

    for entry in oracle_report.get("entries", []) or []:

        qnode_entries[str(entry.get("path"))].append(entry)

    tests = [str(x) for x in profile.get("test_files", []) or []]

    file_rows = []

    for rec in records:

        path = str(rec.get("path"))

        preview = str(rec.get("preview", ""))

        low = preview.lower()

        entries = qnode_entries.get(path, [])

        if not entries and not rec.get("mentions_pennylane"):

            continue

        measurement_families = sorted({m for e in entries for m in (e.get("measurements") or [])})

        loop_risk = sum(1 for e in entries if e.get("has_loop"))

        conditional_risk = sum(1 for e in entries if e.get("uses_conditional"))

        device_mentions = low.count("qml.device") + low.count(".device(") + low.count("default.qubit")

        interface_signals = sum(low.count(tok) for tok in ["interface=", "jax", "torch", "tensorflow", "autograd"])

        shot_signals = low.count("shots=")

        fragility = _clamp01(loop_risk * 0.22 + conditional_risk * 0.18 + interface_signals * 0.04 + shot_signals * 0.08 + (0.06 if len(entries) > 1 else 0.0))

        nearby_tests = [t for t in tests if Path(t).parts[:1] == Path(path).parts[:1] or Path(t).stem in path or Path(path).stem in t]

        benchmark_desert = bool(entries) and not nearby_tests

        row = {

            "path": path,

            "qnode_count": len(entries),

            "measurement_diversity": len(measurement_families),

            "measurement_families": measurement_families,

            "device_signal_count": device_mentions,

            "gradient_fragility": round(fragility, 4),

            "benchmark_desert": benchmark_desert,

            "nearby_tests": nearby_tests[:20],

            "score": round(len(entries) * 0.4 + len(measurement_families) * 0.14 + fragility * 0.28 + (0.18 if benchmark_desert else 0.0), 4),

        }

        file_rows.append(row)

    file_rows.sort(key=lambda x: (-x["score"], x["path"]))

    return {

        "enabled": True,

        "file_count": len(file_rows),

        "hotspots": file_rows[:16],

        "files": file_rows[:120],

        "weather_fronts": [

            {"name": "gradient_fragility_front", "paths": [r["path"] for r in file_rows if r["gradient_fragility"] >= 0.45][:16]},

            {"name": "benchmark_desert_front", "paths": [r["path"] for r in file_rows if r["benchmark_desert"]][:16]},

            {"name": "measurement_diversity_front", "paths": [r["path"] for r in file_rows if r["measurement_diversity"] >= 2][:16]},

        ],

        "device_fragmentation": len({call for call in oracle_report.get("device_calls", []) or []}),

    }

def merge_focus_chunks(*chunk_sets: List[Dict[str, Any]], max_chunks: int = 28) -> List[Dict[str, Any]]:

    merged = []

    seen = set()

    for chunk_list in chunk_sets:

        for chunk in chunk_list or []:

            key = (chunk.get("path"), chunk.get("chunk_id"), sha256_text(str(chunk.get("content", "")))[:16])

            if key in seen:

                continue

            seen.add(key)

            merged.append(chunk)

            if len(merged) >= max_chunks:

                return merged

    return merged

def build_retrieval_constellation(

    repo_dir: Path,

    records: List[Dict[str, Any]],

    repo_brief: Dict[str, Any],

    task_charter: Dict[str, Any],

    weather_map: Dict[str, Any],

    parliament: Dict[str, Any],

) -> Dict[str, Any]:

    if not ENABLE_RETRIEVAL_CONSTELLATION:

        return {"enabled": False, "chunks": []}

    query_parts = [

        task_charter.get("task_title", ""),

        task_charter.get("objective", ""),

        " ".join(task_charter.get("acceptance_criteria", [])[:12]),

        " ".join(task_charter.get("target_files", [])[:20]),

        " ".join(repo_brief.get("recommended_focus_files", [])[:20]),

        " ".join(parliament.get("must_have_actions", [])[:12]),

        " ".join([h.get("path", "") for h in weather_map.get("hotspots", [])[:10]]),

        " ".join(parliament.get("favored_biomes", [])[:6]),

    ]

    query = " ".join([x for x in query_parts if x]).strip()

    chunks = retrieve_focus_chunks(repo_dir, records, query, top_k=max(8, TOP_REPO_CHUNKS))

    return {

        "enabled": True,

        "query": query,

        "chunks": chunks[: TOP_REPO_CHUNKS * 2],

        "hotspot_paths": [h.get("path") for h in weather_map.get("hotspots", [])[:16]],

        "must_have_actions": parliament.get("must_have_actions", [])[:20],

    }

def build_repo_memory_braid(

    profile: Dict[str, Any],

    task_charter: Dict[str, Any],

    capsules: List[Dict[str, Any]],

    harmonic_pack: Dict[str, Any],

    parliament: Dict[str, Any],

    weather_map: Dict[str, Any],

) -> Dict[str, Any]:

    if not ENABLE_REPO_MEMORY_BRAID:

        return {"enabled": False, "strands": []}

    strands = []

    if profile.get("pennylane_files"):

        strands.append({

            "strand": "quantum_shape_guard",

            "theme": "Protect qnode signatures, measurements, devices, and gradient behavior.",

            "anchors": [h.get("path") for h in weather_map.get("hotspots", [])[:8]],

        })

    if profile.get("color_files"):

        strands.append({

            "strand": "chromatic_balance_guard",

            "theme": "Keep color and palette semantics coherent across mentor-guided changes.",

            "anchors": list((harmonic_pack.get("target_palette") or {}).keys()),

        })

    strands.append({

        "strand": "parliament_rationale",

        "theme": "Preserve the reasoning behind biome budgets, cautions, and favored interventions.",

        "anchors": parliament.get("mentor_notes", [])[:8],

    })

    strands.append({

        "strand": "operator_teachability",

        "theme": "Leave the repo easier to own after the arena has finished.",

        "anchors": task_charter.get("acceptance_criteria", [])[:8],

    })

    return {

        "enabled": True,

        "strand_count": len(strands),

        "strands": strands,

        "capsule_ids": [c.get("capsule_id") for c in capsules],

    }

def build_shape_sentinel_script(state: RuntimeState, attempt_index: int, candidate_index: int, oracle_report: Dict[str, Any]) -> Optional[Path]:

    if not ENABLE_SHAPE_SENTINEL:

        return None

    entries = []

    for entry in oracle_report.get("entries", [])[:24]:

        entries.append({

            "path": entry.get("path"),

            "function": entry.get("function"),

            "arg_count": int(entry.get("arg_count", 0) or 0),

            "measurements": entry.get("measurements", []),

        })

    if not entries:

        return None

    script = f"""import importlib, importlib.util, inspect, json, pathlib, sys
ROOT = pathlib.Path.cwd()
ENTRIES = {json.dumps(entries)}
results = []
import_failures = 0
zero_arg_failures = 0
for entry in ENTRIES:
    rel = entry.get("path")
    fn_name = entry.get("function")
    item = { "path": rel, "function": fn_name}
    path = ROOT / rel
    item["exists"] = path.exists()
    if not path.exists():
        import_failures += 1
        item["import_ok"] = False
        item["error"] = "missing_file"
        results.append(item)
        continue
    module_name = pathlib.Path(rel).with_suffix("").as_posix().replace("/", ".")
    mod = None
    try:
        sys.path.insert(0, str(ROOT))
        mod = importlib.import_module(module_name)
        item["import_mode"] = "module"
    except Exception as primary_exc:
        try:
            spec = importlib.util.spec_from_file_location("shape_sentinel_" + module_name.replace(".", "_"), path)
            mod = importlib.util.module_from_spec(spec)
            assert spec and spec.loader
            spec.loader.exec_module(mod)
            item["import_mode"] = "file"
        except Exception as secondary_exc:
            import_failures += 1
            item["import_ok"] = False
            item["error"] = type(secondary_exc).__name__
            item["primary_error"] = type(primary_exc).__name__
            results.append(item)
            continue
    item["import_ok"] = True
    target = getattr(mod, fn_name, None)
    item["callable"] = callable(target)
    if callable(target):
        try:
            item["signature"] = str(inspect.signature(target))
        except Exception:
            item["signature"] = "unavailable"
        if int(entry.get("arg_count", 0) or 0) == 0:
            try:
                value = target()
                item["zero_arg_call_ok"] = True
                item["return_type"] = type(value).__name__
                try:
                    item["return_length"] = len(value)
                except Exception:
                    item["return_length"] = None
            except Exception as call_exc:
                zero_arg_failures += 1
                item["zero_arg_call_ok"] = False
                item["zero_arg_error"] = type(call_exc).__name__
    results.append(item)
report = { "total_entries": len(ENTRIES), "import_failures": import_failures, "zero_arg_failures": zero_arg_failures, "results": results}
print(json.dumps(report))
"""

    path = state.emit_text(f"shape_sentinel/attempt_{attempt_index}_candidate_{candidate_index}.py", script)

    return path

def run_validation_suite_v26(repo_dir: Path, commands: List[str], benchmark_commands: List[str], sentinel_script: Optional[Path]) -> Dict[str, Any]:

    base = run_validation_suite(repo_dir, commands, benchmark_commands)

    if sentinel_script is None:

        base["shape_sentinel"] = {"enabled": False}

        base["shape_sentinel_passed"] = None

        return base

    sentinel_cmd = f'"{sys.executable}" "{sentinel_script}"'

    sentinel_run = run_cmd(sentinel_cmd, cwd=repo_dir, timeout=1800)

    report = {}

    try:

        report = json.loads((sentinel_run.get("stdout") or "").strip() or "{}")

    except Exception:

        report = {"parse_error": True, "stdout_excerpt": clipped(str(sentinel_run.get("stdout", "")), 2000)}

    passed = sentinel_run.get("returncode") == 0 and int(report.get("import_failures", 0) or 0) == 0

    base["shape_sentinel"] = {

        "enabled": True,

        "command": sentinel_run,

        "report": report,

        "passed": passed,

    }

    base["shape_sentinel_passed"] = passed

    if not passed:

        base["passed"] = False

    return base

def biome_alignment_bonus(biome: str, parliament: Dict[str, Any], validation: Dict[str, Any], tags: Dict[str, Any]) -> Dict[str, float]:

    budget = float((parliament.get("patch_budget") or {}).get(biome, 0.0))

    top_budget = max((parliament.get("patch_budget") or {"oasis": 0.0}).values()) if parliament.get("patch_budget") else 0.0

    bonus = 0.0

    penalty = 0.0

    if budget >= top_budget and top_budget > 0:

        bonus += 0.12

    if biome == "oasis" and validation.get("passed"):

        bonus += 0.08

    if biome == "forge" and tags.get("tests", 0) > 0:

        bonus += 0.08

    if biome == "storm" and not validation.get("passed"):

        penalty += 0.20

    if biome == "prism" and tags.get("quantum", 0) > 0 and tags.get("color", 0) > 0:

        bonus += 0.18

    return {"bonus": round(bonus, 4), "penalty": round(penalty, 4), "budget_weight": round(budget, 6)}

def touched_weather_hotspots(diff: Dict[str, Any], weather_map: Dict[str, Any]) -> List[str]:

    touched = set((diff.get("added") or []) + (diff.get("changed") or []) + (diff.get("removed") or []))

    hotspots = [h.get("path") for h in weather_map.get("hotspots", []) if h.get("path")]

    return [p for p in hotspots if p in touched]

def candidate_meta_bonuses_v26(

    diff: Dict[str, Any],

    validation: Dict[str, Any],

    oracle_report: Dict[str, Any],

    harmonic_pack: Dict[str, Any],

    weather_map: Dict[str, Any],

    parliament: Dict[str, Any],

    biome: str,

) -> Dict[str, Any]:

    meta = candidate_meta_bonuses(diff, validation, oracle_report, harmonic_pack)

    extra_bonus = 0.0

    extra_penalty = 0.0

    hotspot_hits = touched_weather_hotspots(diff, weather_map)

    if hotspot_hits:

        extra_bonus += min(0.28, 0.08 * len(hotspot_hits))

    if validation.get("shape_sentinel_passed") is True:

        extra_bonus += 0.35

    biome_meta = biome_alignment_bonus(biome, parliament, validation, meta.get("tags", {}))

    extra_bonus += biome_meta["bonus"]

    extra_penalty += biome_meta["penalty"]

    meta["bonus"] = round(float(meta.get("bonus", 0.0)) + extra_bonus, 4)

    meta["penalty"] = round(float(meta.get("penalty", 0.0)) + extra_penalty, 4)

    meta["weather_hotspots_touched"] = hotspot_hits

    meta["patch_biome"] = biome

    meta["biome_budget_weight"] = biome_meta["budget_weight"]

    meta["shape_sentinel_passed"] = validation.get("shape_sentinel_passed")

    return meta

def summarize_candidate_for_judge_v26(

    patch_plan: Dict[str, Any],

    apply_result: Dict[str, Any],

    diff: Dict[str, Any],

    validation: Dict[str, Any],

    chroma_episode: ChromaticEpisode,

    biome: str,

    meta_score: Dict[str, Any],

    parliament: Dict[str, Any],

) -> Dict[str, Any]:

    base = summarize_candidate_for_judge(

        patch_plan=patch_plan,

        apply_result=apply_result,

        diff=diff,

        validation=validation,

        chroma_episode=chroma_episode,

    )

    base["patch_biome"] = biome

    base["shape_sentinel"] = validation.get("shape_sentinel")

    base["meta_score"] = meta_score

    base["mentor_parliament"] = {

        "mentor_notes": parliament.get("mentor_notes", [])[:10],

        "must_have_actions": parliament.get("must_have_actions", [])[:10],

        "patch_budget": parliament.get("patch_budget", {}),

    }

    return base

def build_patch_payload_v26(

    repo_brief: Dict[str, Any],

    charter: Dict[str, Any],

    execution_graph: Dict[str, Any],

    validation_plan: Dict[str, Any],

    focus_chunks: List[Dict[str, Any]],

    circuit_pack: Dict[str, Any],

    mentor_pack: Dict[str, Any],

    chroma_episode: ChromaticEpisode,

    prior_attempt_notes: List[str],

    oracle_report: Dict[str, Any],

    harmonic_pack: Dict[str, Any],

    benchmark_lab: Dict[str, Any],

    genome: SpectralPatchGenome,

    mentor_parliament: Dict[str, Any],

    circuit_weather_map: Dict[str, Any],

    patch_biome: str,

    retrieval_constellation: Dict[str, Any],

    repo_memory_braid: Dict[str, Any],

    palette_drift_ledger: List[Dict[str, Any]],

) -> str:

    payload = {

        "repo_brief": repo_brief,

        "task_charter": charter,

        "execution_graph": execution_graph,

        "validation_plan": validation_plan,

        "circuit_pack": circuit_pack,

        "mentor_pack": mentor_pack,

        "chromatic_episode": chroma_episode.to_dict(),

        "prior_attempt_notes": prior_attempt_notes[-12:],

        "focus_chunks": compact_focus_chunks(focus_chunks, max_chunks=16),

        "qnode_shape_oracle": oracle_report,

        "color_harmonics": harmonic_pack,

        "benchmark_lab": benchmark_lab,

        "spectral_patch_genome": genome.to_dict(),

        "mentor_parliament": mentor_parliament,

        "circuit_weather_map": {

            "hotspots": circuit_weather_map.get("hotspots", [])[:10],

            "weather_fronts": circuit_weather_map.get("weather_fronts", [])[:6],

        },

        "patch_biome": {

            "name": patch_biome,

            "guidance": PATCH_BIOME_GUIDANCE.get(patch_biome, ""),

        },

        "retrieval_constellation": {

            "query": retrieval_constellation.get("query", ""),

            "hotspot_paths": retrieval_constellation.get("hotspot_paths", [])[:12],

        },

        "repo_memory_braid": repo_memory_braid,

        "palette_drift_ledger_tail": palette_drift_ledger[-12:],

        "advanced_instruction": (

            "Ground every edit in the retrieved repo context. "

            "Respect mentor parliament cautions and the assigned patch biome. "

            "If touching qnodes or PennyLane logic, preserve shape semantics and improve validation. "

            "If touching color logic, keep palette behavior coherent and explainable. "

            "Use the spectral genome as a bias, not as literal output text."

        ),

    }

    return pretty_json(payload)

def arena_breakdown(record: Dict[str, Any]) -> Dict[str, float]:

    judge_score = float((record.get("judge") or {}).get("score_0_to_10", 0.0))

    validation = record.get("validation") or {}

    meta = record.get("meta_score") or {}

    tags = meta.get("tags", {}) or {}

    implementation = min(10.0, judge_score)

    verification = 10.0 if validation.get("passed") else (6.0 if validation.get("shape_sentinel_passed") else 2.0)

    quantum = min(10.0, tags.get("quantum", 0) * 2.0 + (3.0 if validation.get("shape_sentinel_passed") else 0.0) + len(meta.get("weather_hotspots_touched", [])) * 0.5)

    pedagogy = min(10.0, tags.get("docs", 0) * 2.0 + tags.get("tests", 0) * 1.5 + float(meta.get("palette_alignment", 0.0)) * 4.0)

    return {

        "implementation": round(implementation, 4),

        "verification": round(verification, 4),

        "quantum": round(quantum, 4),

        "pedagogy": round(pedagogy, 4),

    }

def materialize_repo_teaching_layer(

    state: RuntimeState,

    llm: BaseLLM,

    final_record: Dict[str, Any],

    mentor_parliament: Dict[str, Any],

    weather_map: Dict[str, Any],

    palette_drift_ledger: List[Dict[str, Any]],

) -> Dict[str, Any]:

    if not ENABLE_REPO_TEACHING_LAYER:

        return {"enabled": False}

    teaching = llm_json_with_artifact(

        state=state,

        llm=llm,

        name="repo_teaching_layer",

        system=REPO_TEACHING_LAYER_SYSTEM,

        user=pretty_json({

            "final_record": final_record,

            "mentor_parliament": mentor_parliament,

            "circuit_weather_map": weather_map,

            "palette_drift_ledger_tail": palette_drift_ledger[-24:],

        }),

        temperature=0.12,

        max_tokens=2200,

    )

    state.emit_text("teaching/operator_notes.md", "\n".join(["# Operator Notes", ""] + [f"- {x}" for x in teaching.get("operator_notes", [])]))

    state.emit_text("teaching/migration_hints.md", "\n".join(["# Migration Hints", ""] + [f"- {x}" for x in teaching.get("migration_hints", [])]))

    state.emit_json("teaching/failure_capsules.json", teaching.get("failure_capsules", []))

    state.emit_text("teaching/readme_delta_recommendations.md", "\n".join(["# README Delta Recommendations", ""] + [f"- {x}" for x in teaching.get("readme_delta_recommendations", [])]))

    state.emit_text("teaching/own_this_repo_summary.md", "\n".join(["# Own This Repo Summary", "", str(teaching.get("own_this_repo_summary", "")), "", "## Follow-on Tasks", ""] + [f"- {x}" for x in teaching.get("follow_on_tasks", [])]))

    teaching["enabled"] = True

    return teaching


In [ ]:
from __future__ import annotations

def run_agentic_repo_runtime_v26() -> Dict[str, Any]:

    state = RuntimeState(ensure_dir(WORK_ROOT / f"{safe_slug(RUN_LABEL, 'repo_runtime_v26')}_{now_ts()}"))

    state.log_event("runtime_start", version="v26")

    llm = build_llm()

    repo_dir = materialize_repo(state)

    branch_res = maybe_create_branch(repo_dir, f"agentic/{safe_slug(RUN_LABEL, 'runtime-v26')}-{now_ts()}")

    if branch_res:

        state.emit_json("git_branch_result.json", branch_res)

    records = repo_file_records(repo_dir, max_file_chars=MAX_FILE_CHARS)

    profile = infer_repo_profile(repo_dir, records)

    index = build_repo_index(records)

    state.emit_json("repo_manifest.json", records[:8000])

    state.emit_json("repo_profile.json", profile)

    state.emit_json("repo_index.json", index[:8000])

    bootstrap_chunks = retrieve_focus_chunks(repo_dir, records, TASK_PROMPT or ISSUE_CONTEXT or "repo overview quantum color architecture benchmark", top_k=TOP_REPO_CHUNKS)

    context_pack = repo_context_text(profile, index, bootstrap_chunks)

    state.emit_text("repo_context_pack.txt", context_pack)

    repo_brief = llm_json_with_artifact(state=state, llm=llm, name="repo_brief", system=REPO_BRIEF_SYSTEM, user=context_pack, temperature=0.15, max_tokens=1500)

    charter_prompt = pretty_json({"repo_brief": repo_brief, "repo_profile": profile, "user_task_prompt": TASK_PROMPT, "issue_context": ISSUE_CONTEXT, "instruction": "If user_task_prompt is blank, invent one valuable task grounded in this repo."})

    task_charter = llm_json_with_artifact(state=state, llm=llm, name="task_charter", system=TASK_CHARTER_SYSTEM, user=charter_prompt, temperature=0.2, max_tokens=1800)

    focus_text = " ".join([task_charter.get("task_title", ""), task_charter.get("objective", ""), " ".join(task_charter.get("acceptance_criteria", [])[:20]), " ".join(task_charter.get("target_files", [])[:40])])

    focus_chunks = retrieve_focus_chunks(repo_dir, records, focus_text, top_k=TOP_REPO_CHUNKS)

    state.emit_json("focus_chunks_initial.json", focus_chunks)

    execution_graph = llm_json_with_artifact(state=state, llm=llm, name="execution_graph", system=EXECUTION_GRAPH_SYSTEM, user=pretty_json({"repo_brief": repo_brief, "task_charter": task_charter, "focus_chunks": compact_focus_chunks(focus_chunks, max_chunks=10)}), temperature=0.2, max_tokens=1800)

    validation_plan = llm_json_with_artifact(state=state, llm=llm, name="validation_plan", system=VALIDATION_PLAN_SYSTEM, user=pretty_json({"repo_profile": profile, "task_charter": task_charter, "repo_brief": repo_brief, "focus_chunks": compact_focus_chunks(focus_chunks, max_chunks=8)}), temperature=0.1, max_tokens=1500)

    mentor_pack = llm_json_with_artifact(state=state, llm=llm, name="chromatic_mentor_bootstrap", system=COLOR_MENTOR_SYSTEM, user=pretty_json({"repo_profile": profile, "task_charter": task_charter, "repo_brief": repo_brief, "execution_graph": execution_graph}), temperature=0.25, max_tokens=1200)

    circuit_pack = llm_json_with_artifact(state=state, llm=llm, name="circuit_arena_bootstrap", system=CIRCUIT_ARENA_SYSTEM, user=pretty_json({"repo_profile": profile, "task_charter": task_charter, "focus_chunks": compact_focus_chunks([c for c in focus_chunks if c.get("mentions_pennylane")], max_chunks=10)}), temperature=0.2, max_tokens=1800)

    oracle_report = build_qnode_shape_oracle(repo_dir, profile, records)

    state.emit_json("qnode_shape_oracle.json", oracle_report)

    harmonic_pack = derive_color_harmonics(profile, task_charter, focus_chunks)

    state.emit_json("chromatic_harmonics.json", harmonic_pack)

    benchmark_lab = llm_json_with_artifact(state=state, llm=llm, name="benchmark_lab", system=BENCHMARK_LAB_SYSTEM, user=pretty_json({"repo_profile": profile, "task_charter": task_charter, "validation_plan": validation_plan, "qnode_shape_oracle": oracle_report, "color_harmonics": harmonic_pack, "focus_chunks": compact_focus_chunks(focus_chunks, max_chunks=8)}), temperature=0.12, max_tokens=1600)

    mentor_parliament = build_mentor_parliament(state=state, llm=llm, repo_profile=profile, repo_brief=repo_brief, task_charter=task_charter, execution_graph=execution_graph, color_harmonics=harmonic_pack, qnode_shape_oracle=oracle_report, benchmark_lab=benchmark_lab)

    state.emit_json("mentor_parliament.json", mentor_parliament)

    circuit_weather_map = build_circuit_weather_map(repo_dir, profile, records, oracle_report)

    state.emit_json("circuit_weather_map.json", circuit_weather_map)

    retrieval_constellation = build_retrieval_constellation(repo_dir=repo_dir, records=records, repo_brief=repo_brief, task_charter=task_charter, weather_map=circuit_weather_map, parliament=mentor_parliament)

    state.emit_json("retrieval_constellation.json", {**retrieval_constellation, "chunks": compact_focus_chunks(retrieval_constellation.get("chunks", []), max_chunks=18)})

    focus_chunks = merge_focus_chunks(focus_chunks, retrieval_constellation.get("chunks", []), max_chunks=max(24, TOP_REPO_CHUNKS * 2))

    state.emit_json("focus_chunks.json", focus_chunks)

    genome_bootstrap = llm_json_with_artifact(state=state, llm=llm, name="spectral_genome_bootstrap", system=SPECTRAL_GENOME_SYSTEM, user=pretty_json({"repo_profile": profile, "task_charter": task_charter, "repo_brief": repo_brief, "qnode_shape_oracle": oracle_report, "color_harmonics": harmonic_pack, "benchmark_lab": benchmark_lab, "mentor_parliament": mentor_parliament, "circuit_weather_map": circuit_weather_map}), temperature=0.15, max_tokens=1400)

    capsules = chromatic_memory_capsules(profile, task_charter)

    state.emit_json("memory_capsules.json", capsules)

    repo_memory_braid = build_repo_memory_braid(profile, task_charter, capsules, harmonic_pack, mentor_parliament, circuit_weather_map)

    state.emit_json("repo_memory_braid.json", repo_memory_braid)

    commands = select_validation_commands(profile, validation_plan)

    benchmark_commands = select_benchmark_commands(profile, validation_plan)

    for cmd in benchmark_lab.get("benchmark_commands", []):

        if isinstance(cmd, str) and cmd.strip() and normalize_ws(cmd) not in {normalize_ws(x) for x in benchmark_commands}:

            benchmark_commands.append(cmd)

    benchmark_commands = benchmark_commands[:6]

    state.emit_json("resolved_validation_commands.json", {"commands": commands, "benchmark_commands": benchmark_commands})

    base_repo = state.snapshot_repo(repo_dir, "base_repo", ignore_git=False)

    best_record = None

    attempt_notes = []

    all_candidates = []

    palette_drift_ledger = []

    current_genome = SpectralPatchGenome.seeded_from_context(profile=profile, task_charter=task_charter, harmonic_pack=harmonic_pack, oracle_report=oracle_report, bootstrap=genome_bootstrap)

    state.emit_json("spectral_genome_seed.json", current_genome.to_dict())

    genome_history = [{"phase": "seed", "genome": current_genome.to_dict()}]

    for attempt_idx in range(1, MAX_ATTEMPTS + 1):

        mentor_attempt = llm_json_with_artifact(state=state, llm=llm, name=f"chromatic_mentor_attempt_{attempt_idx}", system=COLOR_MENTOR_SYSTEM, user=pretty_json({"attempt_index": attempt_idx, "task_charter": task_charter, "prior_attempt_notes": attempt_notes[-12:], "capsules": capsules, "repo_profile": profile, "color_harmonics": harmonic_pack, "mentor_parliament": mentor_parliament}), temperature=0.25, max_tokens=1200)

        parliament_attempt = build_mentor_parliament(state=state, llm=llm, repo_profile=profile, repo_brief=repo_brief, task_charter=task_charter, execution_graph=execution_graph, color_harmonics=harmonic_pack, qnode_shape_oracle=oracle_report, benchmark_lab=benchmark_lab, prior_attempt_notes=attempt_notes, name=f"mentor_parliament_attempt_{attempt_idx}")

        circuit_attempt = llm_json_with_artifact(state=state, llm=llm, name=f"circuit_arena_attempt_{attempt_idx}", system=CIRCUIT_ARENA_SYSTEM, user=pretty_json({"attempt_index": attempt_idx, "repo_profile": profile, "task_charter": task_charter, "prior_attempt_notes": attempt_notes[-12:], "focus_chunks": compact_focus_chunks([c for c in focus_chunks if c.get("mentions_pennylane")], max_chunks=10), "qnode_shape_oracle": oracle_report, "circuit_weather_map": circuit_weather_map}), temperature=0.2, max_tokens=1800)

        attempt_genome_guidance = llm_json_with_artifact(state=state, llm=llm, name=f"spectral_genome_attempt_{attempt_idx}", system=SPECTRAL_GENOME_SYSTEM, user=pretty_json({"attempt_index": attempt_idx, "current_genome": current_genome.to_dict(), "best_so_far": best_record or {}, "task_charter": task_charter, "qnode_shape_oracle": oracle_report, "color_harmonics": harmonic_pack, "attempt_notes": attempt_notes[-12:], "mentor_parliament": parliament_attempt}), temperature=0.12, max_tokens=1400)

        attempt_seed = SpectralPatchGenome.blend([current_genome, SpectralPatchGenome.from_dict(attempt_genome_guidance.get("genome_bias", {}))])

        genome_history.append({"phase": f"attempt_{attempt_idx}_pre", "genome": attempt_seed.to_dict()})

        for candidate_idx in range(1, CANDIDATES_PER_ATTEMPT + 1):

            biome = assigned_biome(attempt_idx, candidate_idx)

            candidate_repo = state.snapshot_repo(base_repo, f"attempt_{attempt_idx}_candidate_{candidate_idx}", ignore_git=False)

            chroma_episode = build_chromatic_episode(attempt_index=attempt_idx, candidate_index=candidate_idx, mentor_notes=(mentor_attempt.get("mentor_notes", []) + parliament_attempt.get("mentor_notes", []))[:24], has_quantum=bool(profile.get("pennylane_files")), has_color=bool(profile.get("color_files")))

            state.emit_json(f"chromatic/attempt_{attempt_idx}_candidate_{candidate_idx}.json", chroma_episode.to_dict())

            candidate_genome = attempt_seed.mutate(attempt_index=attempt_idx, candidate_index=candidate_idx, mutation_rate=GENOME_MUTATION_RATE, harmonic_pack=harmonic_pack, oracle_report=oracle_report)

            state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/spectral_genome.json", candidate_genome.to_dict())

            patch_plan = llm_json_with_artifact(state=state, llm=llm, name=f"patch_plan_attempt_{attempt_idx}_candidate_{candidate_idx}", system=PATCH_CANDIDATE_SYSTEM, user=build_patch_payload_v26(repo_brief=repo_brief, charter=task_charter, execution_graph=execution_graph, validation_plan=validation_plan, focus_chunks=focus_chunks, circuit_pack=circuit_attempt if circuit_attempt.get("enabled") else circuit_pack, mentor_pack=mentor_attempt, chroma_episode=chroma_episode, prior_attempt_notes=attempt_notes, oracle_report=oracle_report, harmonic_pack=harmonic_pack, benchmark_lab=benchmark_lab, genome=candidate_genome, mentor_parliament=parliament_attempt, circuit_weather_map=circuit_weather_map, patch_biome=biome, retrieval_constellation=retrieval_constellation, repo_memory_braid=repo_memory_braid, palette_drift_ledger=palette_drift_ledger), temperature=0.18 + (candidate_idx - 1) * 0.07, max_tokens=3400)

            patch_plan["patch_biome"] = biome

            patch_plan["biome_guidance"] = PATCH_BIOME_GUIDANCE.get(biome, "")

            state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/patch_plan_materialized.json", patch_plan)

            apply_result = apply_patch_plan(candidate_repo, patch_plan)

            state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/apply_result.json", apply_result)

            commit_res = maybe_commit(candidate_repo, f"attempt {attempt_idx} candidate {candidate_idx} {biome}")

            if commit_res:

                state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/git_commit_result.json", commit_res)

            diff = repo_diff_summary(base_repo, candidate_repo)

            state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/diff_summary.json", {k: v for k, v in diff.items() if k != "patches"})

            state.emit_text(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/patches.diff", "\n\n".join([v for _, v in sorted(diff.get("patches", {}).items())])[:200000])

            merged_commands = commands[:]

            for cmd in patch_plan.get("tests_to_run", []):

                if isinstance(cmd, str) and cmd.strip() and normalize_ws(cmd) not in {normalize_ws(x) for x in merged_commands}:

                    merged_commands.append(cmd)

            sentinel_script = build_shape_sentinel_script(state, attempt_idx, candidate_idx, oracle_report)

            validation = run_validation_suite_v26(candidate_repo, merged_commands[:6], benchmark_commands, sentinel_script)

            state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/validation.json", validation)

            meta_score = candidate_meta_bonuses_v26(diff, validation, oracle_report, harmonic_pack, circuit_weather_map, parliament_attempt, biome)

            state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/meta_score.json", meta_score)

            palette_drift_ledger.append({"attempt_index": attempt_idx, "candidate_index": candidate_idx, "patch_biome": biome, "palette_alignment": meta_score.get("palette_alignment"), "weather_hotspots_touched": meta_score.get("weather_hotspots_touched", [])})

            judge_input = summarize_candidate_for_judge_v26(patch_plan=patch_plan, apply_result=apply_result, diff=diff, validation=validation, chroma_episode=chroma_episode, biome=biome, meta_score=meta_score, parliament=parliament_attempt)

            judge = llm_json_with_artifact(state=state, llm=llm, name=f"judge_attempt_{attempt_idx}_candidate_{candidate_idx}", system=JUDGE_SYSTEM, user=pretty_json({"task_charter": task_charter, "acceptance_criteria": task_charter.get("acceptance_criteria", []), "candidate": judge_input, "qnode_shape_oracle": oracle_report, "color_harmonics": harmonic_pack, "benchmark_lab": benchmark_lab, "spectral_patch_genome": candidate_genome.to_dict(), "mentor_parliament": parliament_attempt, "circuit_weather_map": circuit_weather_map}), temperature=0.1, max_tokens=1600)

            score = float(judge.get("score_0_to_10", 0.0))

            if validation.get("passed"):

                score += 1.25

            if validation.get("passed_benchmarks") is True:

                score += 0.5

            if validation.get("shape_sentinel_passed") is True:

                score += 0.35

            if apply_result.get("error_count", 0) > 0:

                score -= 1.75

            score += float(meta_score.get("bonus", 0.0))

            score -= float(meta_score.get("penalty", 0.0))

            record = {"attempt_index": attempt_idx, "candidate_index": candidate_idx, "patch_biome": biome, "candidate_repo": str(candidate_repo), "patch_plan": patch_plan, "apply_result": apply_result, "diff": {k: v for k, v in diff.items() if k != "patches"}, "validation": validation, "judge": judge, "chromatic_episode": chroma_episode.to_dict(), "genome": candidate_genome.to_dict(), "meta_score": meta_score, "arena_breakdown": {}, "composite_score": round(score, 4)}

            record["arena_breakdown"] = arena_breakdown(record)

            state.emit_json(f"attempts/attempt_{attempt_idx}/candidate_{candidate_idx}/candidate_record.json", record)

            all_candidates.append(record)

        ranked_global = sorted(all_candidates, key=lambda r: (-r["composite_score"], r["attempt_index"], r["candidate_index"]))

        best_record = ranked_global[0]

        state.emit_json(f"attempts/attempt_{attempt_idx}/arena_of_arenas.json", {"attempt_index": attempt_idx, "winner_attempt": best_record["attempt_index"], "winner_candidate": best_record["candidate_index"], "winner_score": best_record["composite_score"], "winner_biome": best_record["patch_biome"], "top_6": [{"attempt_index": r["attempt_index"], "candidate_index": r["candidate_index"], "patch_biome": r["patch_biome"], "composite_score": r["composite_score"], "verdict": r["judge"].get("verdict"), "arena_breakdown": r.get("arena_breakdown", {}), "weather_hotspots_touched": r["meta_score"].get("weather_hotspots_touched", [])} for r in ranked_global[:6]]})

        selected_for_blend = [genome_feedback_from_record(r) for r in ranked_global[:max(1, GENOME_SELECTION_TOP_K)]]

        current_genome = SpectralPatchGenome.blend(selected_for_blend)

        genome_history.append({"phase": f"attempt_{attempt_idx}_post", "genome": current_genome.to_dict()})

        attempt_notes.append(f"Attempt {attempt_idx} best candidate was A{best_record['attempt_index']}C{best_record['candidate_index']} with score {best_record['composite_score']}. Verdict={best_record['judge'].get('verdict')} Biome={best_record['patch_biome']} PaletteAlignment={best_record['meta_score'].get('palette_alignment')}.")

        attempt_notes.extend(best_record["judge"].get("weaknesses", [])[:4])

        if best_record["validation"].get("passed") and best_record["validation"].get("shape_sentinel_passed") in {True, None} and str(best_record["judge"].get("acceptance_pass_likelihood", "")).lower() in {"high", "very high"}:

            break

        base_repo = Path(best_record["candidate_repo"])

    if best_record is None:

        raise RuntimeError("No candidate records were produced.")

    state.emit_json("spectral_genome_history.json", genome_history)

    state.emit_json("palette_drift_ledger.json", palette_drift_ledger)

    final_repo = Path(best_record["candidate_repo"])

    copytree_durable(final_repo, state.best_repo_dir, ignore_git=False)

    final_zip = zip_dir(state.best_repo_dir, state.root / f"{safe_slug(RUN_LABEL, 'repo_runtime_v26')}_final_repo.zip")

    final_record = {"repo_source": REPO_SOURCE, "repo_ref": REPO_REF, "task_prompt": TASK_PROMPT, "issue_context": ISSUE_CONTEXT, "repo_profile": profile, "repo_brief": repo_brief, "task_charter": task_charter, "execution_graph": execution_graph, "validation_plan": validation_plan, "benchmark_lab": benchmark_lab, "qnode_shape_oracle": oracle_report, "chromatic_harmonics": harmonic_pack, "mentor_parliament": mentor_parliament, "circuit_weather_map": circuit_weather_map, "retrieval_constellation": {"query": retrieval_constellation.get("query"), "hotspot_paths": retrieval_constellation.get("hotspot_paths", [])}, "repo_memory_braid": repo_memory_braid, "spectral_genome_history": genome_history, "palette_drift_ledger": palette_drift_ledger, "circuit_bootstrap": circuit_pack, "chromatic_bootstrap": mentor_pack, "best_candidate": best_record, "all_candidate_count": len(all_candidates), "artifacts_count": len(state.artifacts), "final_repo": str(state.best_repo_dir), "final_zip": str(final_zip)}

    teaching_layer = materialize_repo_teaching_layer(state, llm, final_record, mentor_parliament, circuit_weather_map, palette_drift_ledger)

    final_record["repo_teaching_layer"] = teaching_layer

    state.emit_json("final_report.json", final_record)

    materialize_pr_summary(state, llm, final_record)

    summary = {"run_root": str(state.root), "artifacts_dir": str(state.artifacts_dir), "candidate_runs_dir": str(state.candidates_dir), "best_repo_dir": str(state.best_repo_dir), "final_zip": str(final_zip), "task_title": task_charter.get("task_title"), "best_composite_score": best_record["composite_score"], "best_attempt": best_record["attempt_index"], "best_candidate": best_record["candidate_index"], "best_biome": best_record["patch_biome"], "validation_passed": best_record["validation"].get("passed"), "shape_sentinel_passed": best_record["validation"].get("shape_sentinel_passed"), "judge_verdict": best_record["judge"].get("verdict"), "judge_acceptance_pass_likelihood": best_record["judge"].get("acceptance_pass_likelihood"), "pennylane_files_detected": len(profile.get("pennylane_files", [])), "color_files_detected": len(profile.get("color_files", [])), "qnode_count_detected": int(oracle_report.get("qnode_count", 0)), "artifact_count": len(state.artifacts)}

    state.emit_json("summary.json", summary)

    state.log_event("runtime_complete", **summary)

    return summary


In [ ]:
summary = run_agentic_repo_runtime_v26()

print(json.dumps(summary, indent=2))

print(f"\nFinal zip: {summary['final_zip']}")

print(f"Best repo: {summary['best_repo_dir']}")

print(f"Artifacts: {summary['artifacts_dir']}")
